<a href="https://colab.research.google.com/github/arielax-212/Corpus2GeoRAG/blob/main/CollectCorpus/CollectCorpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CollectCorpus: the Russia–Ukraine war corpus from Wikipedia
**Ariel Lax** | Corpus2GeoRAG

Produces **`DATA.json`** for the next step (`ApplyNER`), in exactly the format of `examples/DATA.json`:
`{page title: page text}`, military content only.

```
build ──► update ──► filter ──► export ──► DATA.json ──► ApplyNER ──► ...
```

| Part | | What it does |
|---|---|---|
| 1 | Wikipedia access | Requests to Wikipedia, with rate-limit handling |
| 2 | Build | Choose pages (category trees + seed lists) and download them to `pages.jsonl` |
| 3 | Update | Re-download only the pages that changed on Wikipedia |
| 4 | Filter | Keep military content, by the category decisions |
| 5 | Export | Write `DATA.json` in the original format |
| 6 | **Run** | A quick demo, and the full pipeline |

Parts 1–5 only define functions; everything runs in Part 6.

## Setup

In [1]:
%pip install -q wikipedia-api

In [2]:
import csv
import json
import os
import re
import requests
import sys
import threading
import time
import wikipediaapi
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

---
# Part 1: Wikipedia access
We use **`wikipedia-api`** (not the course's `wikipedia` library, unmaintained since 2014 and blocked by
Wikipedia with HTTP 429). For revision ids, coordinates and the raw page text we call the MediaWiki API directly.

#### Constants
Our User-Agent (required by Wikipedia), titles per request, and the minimum gap between requests.

In [3]:
USER_AGENT = "GeoAI-WikiCorpus/0.1 (student course project)"
BATCH = 50
MIN_INTERVAL = 0.25
session = requests.Session()
session.headers["User-Agent"] = USER_AGENT
_throttle_lock = threading.Lock()
_last_request = 0.0

#### `client()`
A `wikipedia-api` client for a language.

In [4]:
def client(lang="en"):
    return wikipediaapi.Wikipedia(user_agent=USER_AGENT, language=lang)

#### `_throttle()`
Keeps a minimum gap between requests, even in parallel, so Wikipedia doesn't block us.

In [5]:
def _throttle():
    """Keep at least MIN_INTERVAL seconds between our API requests, across threads."""
    global _last_request
    with _throttle_lock:
        wait = _last_request + MIN_INTERVAL - time.time()
        if wait > 0:
            time.sleep(wait)
        _last_request = time.time()

#### `api()`
A MediaWiki API call. On a block (HTTP 429) or a dropped connection it waits, longer each time, and retries.

In [6]:
def api(lang="en", retries=8, **params):
    """Direct call to the MediaWiki API. Waits and retries when rate-limited (HTTP 429)."""
    params = {"action": "query", "format": "json", "formatversion": 2, **params}
    for attempt in range(retries):
        _throttle()
        try:
            r = session.get(f"https://{lang}.wikipedia.org/w/api.php", params=params, timeout=30)
        except requests.ConnectionError:  # Wikipedia sometimes drops the connection under load
            if attempt == retries - 1:
                raise
            time.sleep(5 * (attempt + 1))
            continue
        if r.status_code != 429 or attempt == retries - 1:
            r.raise_for_status()
            return r.json()
        # Retry-After is the server's minimum; back off further on repeated refusals
        time.sleep(max(int(r.headers.get("Retry-After", 0)), 5) * (attempt + 1))

#### `flatten_sections()`
Turns a page's section tree into a list where each section keeps its path, e.g. `History > World War II`.

In [7]:
def flatten_sections(sections, path=()):
    """Turn the section tree into a flat list: [{"path": "History > Kievan Rus'", "text": ...}]."""
    out = []
    for s in sections:
        p = (*path, s.title)
        if s.text.strip():
            out.append({"path": " > ".join(p), "level": len(p), "text": s.text})
        out += flatten_sections(s.sections, p)
    return out

#### `raw_content()`
The page text **exactly** as Wikipedia returns it (`== Heading ==`, empty sections kept). This is the text that goes into `DATA.json`.

In [8]:
def raw_content(title, lang="en"):
    """The page text exactly as Wikipedia returns it: headings as "== X ==", empty sections kept.
    This is the format of the team's examples/DATA.json (the old `wikipedia` library's page.content),
    which `wikipedia-api` can't reproduce because it splits the text into sections."""
    data = api(lang, prop="extracts", explaintext=1, titles=title, redirects=1)
    pages = data["query"]["pages"]
    return pages[0].get("extract", "") if pages else ""

#### `page_meta()`
Revision id, last edit and coordinates for up to 50 pages per request. Follows renames; `None` = page doesn't exist.

In [9]:
def page_meta(titles, lang="en"):
    """Current version info for many pages, 50 per request.

    Returns {requested_title: meta | None}. None = page doesn't exist (deleted / never existed).
    meta = {"title", "pageid", "revid", "last_edited", "coordinates"}; "title" is the canonical
    title after normalization and redirects, so if it differs from the requested one the page was renamed.
    """
    out = {}
    for i in range(0, len(titles), BATCH):
        batch = titles[i:i + BATCH]
        params = {"prop": "info|revisions|coordinates", "rvprop": "ids|timestamp",
                  "colimit": "max", "redirects": 1, "titles": "|".join(batch)}
        renamed, pages, cont = {}, {}, {}
        while True:  # the API may split the answer into several "continue" parts
            data = api(lang, **params, **cont)
            q = data.get("query", {})
            for r in q.get("normalized", []) + q.get("redirects", []):
                renamed[r["from"]] = r["to"]
            for p in q.get("pages", []):
                m = pages.setdefault(p["title"], {"title": p["title"], "coordinates": None})
                if p.get("missing") or p.get("invalid"):
                    m["missing"] = True
                    continue
                m["pageid"] = p["pageid"]
                if p.get("revisions"):
                    m["revid"] = p["revisions"][0]["revid"]
                    m["last_edited"] = p["revisions"][0]["timestamp"]
                if p.get("coordinates"):
                    c = p["coordinates"][0]
                    m["coordinates"] = {"lat": c["lat"], "lon": c["lon"]}
            if "continue" not in data:
                break
            cont = data["continue"]
        for t in batch:
            final = t
            while final in renamed:  # normalized -> redirect target
                final = renamed[final]
            m = pages.get(final)
            out[t] = None if m is None or m.get("missing") else m
    return out

---
# Part 2: Build

**Choosing pages.** A Wikipedia *category* is a tag on a page; categories contain pages and subcategories,
forming a tree. We walk the war's category trees and add seed lists:

| Source | Why |
|---|---|
| `Russo-Ukrainian war`, `Russo-Ukrainian war (2022–present)`, depth 2 | The main category trees of the war |
| Military operations / units / equipment, depth 3 | Battles sit at level 3 (*Military operations → Battles → by year → Battles … in 2022*) |
| `Battles of the war in Donbas`, `Oblasts of Ukraine` | 2014–2021 battles, the 25 oblasts |
| `seeds/team_DATA_json_titles.txt` | The titles of `examples/DATA.json` |
| `seeds/recommended_forces_battles_places.txt` | Pages the corpus links to most but didn't contain: armed forces, battles, places |

**Downloading.** Titles → versions (`page_meta`, 50 per request) → pages in parallel, each appended to
`pages.jsonl` right away, so an interrupted run resumes where it stopped.
`pages.jsonl` keeps everything: sections, coordinates, revisions, and the raw text (`content`).

#### Constants
Wikipedia maintenance categories (e.g. *All articles with dead links*), removed from each page.

In [10]:
NOISE_CATEGORY_PREFIXES = (
    "Category:All ", "Category:Articles ", "Category:CS1", "Category:Webarchive ",
    "Category:Wikipedia ", "Category:Use ", "Category:Short description", "Category:Pages ",
    "Category:Commons ", "Category:Coordinates ", "Category:Official website",
)

#### `corpus_dir()`
Where a corpus is saved: `data/<name>/`.

In [11]:
def corpus_dir(name):
    return Path("data") / name

#### `load_pages()`
Reads `pages.jsonl` into `{title: record}`.

In [12]:
def load_pages(path):
    """pages.jsonl -> {title: record}, in file order."""
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return {r["title"]: r for r in map(json.loads, filter(str.strip, f))}

#### `parse_category()`
`"Category:X@3"` → `("Category:X", 3)`: a category can carry its own depth.

In [13]:
def parse_category(spec, default_depth):
    """"Category:X@3" -> ("Category:X", 3); "Category:X" -> ("Category:X", default_depth)."""
    name, sep, depth = spec.rpartition("@")
    return (name, int(depth)) if sep and depth.isdigit() else (spec, default_depth)

#### `collect_titles()`
Walks each category tree down to its depth and collects page titles, remembering where each was found (`source`).

In [14]:
def collect_titles(wiki, categories, seeds):
    """BFS over each category tree down to its own depth.

    categories: [(name, depth), ...]; seeds: {title: source}. Returns {title: source},
    where source is the category the page was found in (or the seed label).
    Each root gets its own BFS, so a branch reached shallowly from one root is still
    walked to full depth from another root that asks for more.
    """
    titles = dict(seeds)
    members = {}  # category -> its members, so shared subcategories are fetched once
    for root, depth in categories:
        seen, frontier = set(), [root]
        for d in range(depth + 1):
            nxt = []
            for cat in frontier:
                if cat in seen:
                    continue
                seen.add(cat)
                if cat not in members:
                    members[cat] = wiki.page(cat).categorymembers
                for name, member in members[cat].items():
                    if member.ns == 0:
                        titles.setdefault(name, cat)
                    elif member.ns == 14 and d < depth:
                        nxt.append(name)
            frontier = nxt
        print(f"  {root} (depth {depth}): {len(seen)} categories, {len(titles)} pages so far")
    return titles

#### `titles_from_config()`
Seed pages + seed files (one title per line, `#` = comment) + categories.

In [15]:
def titles_from_config(wiki, config):
    """Collect titles using a saved config.json (shared by build and update)."""
    seeds = {t: "seed" for t in config["seeds"]}
    for path in config.get("seed_files", []):
        for t in Path(path).read_text(encoding="utf-8").splitlines():
            t = t.strip()
            if t and not t.startswith("#"):  # blank lines and "# comments" are ignored
                seeds.setdefault(t, f"seed-file:{Path(path).name}")
    categories = [parse_category(c, config["depth"]) for c in config["categories"]]
    return collect_titles(wiki, categories, seeds)

#### `fetch_page()`
Downloads one page into a `pages.jsonl` record.

In [16]:
def fetch_page(wiki, meta, source, with_links):
    """Download one page. `meta` comes from page_meta() and carries revid + coordinates."""
    page = wiki.page(meta["title"])
    return {
        "title": meta["title"],
        "pageid": meta["pageid"],
        "revid": meta["revid"],
        "last_edited": meta["last_edited"],
        "url": page.fullurl,
        "lang": wiki.language,
        "source": source,
        "coordinates": meta["coordinates"],
        "summary": page.summary,
        "sections": flatten_sections(page.sections),
        "text": page.text,
        "content": raw_content(meta["title"], wiki.language),  # exact DATA.json format, see export_corpus.py
        "categories": [c for c in page.categories if not c.startswith(NOISE_CATEGORY_PREFIXES)],
        "links": list(page.links) if with_links else [],
        "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

#### `fetch_many()`
Downloads pages in parallel; one failing page doesn't stop the run.

In [17]:
def fetch_many(wiki, items, with_links, workers, on_record):
    """Download [(meta, source), ...] in parallel; calls on_record(rec) for each page (in one thread)."""
    count, start = 0, time.time()
    with ThreadPoolExecutor(workers) as pool:
        futures = {pool.submit(fetch_page, wiki, m, s, with_links): m["title"] for m, s in items}
        for fut in as_completed(futures):
            try:
                rec = fut.result()
            except Exception as e:  # one bad page shouldn't kill a long run
                print(f"  ! {futures[fut]}: {e}")
                continue
            on_record(rec)
            count += 1
            if count % 25 == 0:
                print(f"  {count}/{len(items)}  ({time.time() - start:.0f}s)")
    return count

#### `build()`
The whole build. Saves `config.json` so `update()` repeats the same choice.

In [18]:
def build(name, categories=(), seeds=(), seed_files=(), depth=1, lang="en", limit=0, workers=4,
          with_links=True):
    """Collect titles, check their versions, download the pages not saved yet -> data/<name>/."""
    out_dir = corpus_dir(name)
    out_dir.mkdir(parents=True, exist_ok=True)
    pages_path = out_dir / "pages.jsonl"
    config = {
        "categories": list(categories), "depth": depth, "seeds": list(seeds),
        "seed_files": list(seed_files), "lang": lang, "with_links": with_links,
    }
    (out_dir / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    wiki = client(lang)

    print("1) collecting titles")
    titles = titles_from_config(wiki, config)
    (out_dir / "titles.txt").write_text("\n".join(sorted(titles)), encoding="utf-8")

    done = set(load_pages(pages_path))
    todo = [t for t in titles if t not in done]
    if limit:
        todo = todo[: limit]

    print(f"2) checking versions of {len(todo)} pages")
    items, seen = [], set(done)
    for title, meta in page_meta(todo, lang).items():
        if meta and meta["title"] not in seen:  # skip missing pages and redirects to pages we have
            seen.add(meta["title"])
            items.append((meta, titles[title]))

    print(f"3) fetching {len(items)} pages ({len(done)} already saved)")
    lock = threading.Lock()
    with pages_path.open("a", encoding="utf-8") as f:
        def save(rec):
            with lock:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                f.flush()
        count = fetch_many(wiki, items, with_links, workers, save)

    print(f"done: {count} new pages -> {pages_path} (total {len(done) + count})")

---
# Part 3: Update
Asks Wikipedia for the current revision of every saved page (50 per request) and re-downloads **only the pages
that changed**. Renamed pages are re-fetched under the new name, deleted ones are marked `missing`,
and every change is logged to `changes.jsonl`.

#### Constants

In [19]:
CHECKPOINT = 200

#### `parse_time()`
Makes Wikipedia's and Python's time formats comparable.

In [20]:
def parse_time(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))

#### `backfill()`
Gives older records a revision id without extra requests.

In [21]:
def backfill(pages, meta):
    """Give old records (saved without revid) a revid, without extra requests:
    if the page's last edit is older than our fetch, what we saved *is* the current revision.
    Otherwise leave revid=None, so the page counts as changed and gets re-downloaded."""
    count = 0
    for title, rec in pages.items():
        if "revid" in rec:
            continue
        m = meta[title]
        if m and m["title"] == title and parse_time(m["last_edited"]) <= parse_time(rec["fetched_at"]):
            rec["revid"], rec["last_edited"] = m["revid"], m["last_edited"]
        else:
            rec["revid"] = None
        count += 1
    return count

#### `try_raw_content()`
`raw_content()` for one page; on failure the page is skipped and retried next run.

In [22]:
def try_raw_content(title, lang):
    """raw_content(), but one failing page doesn't stop a long run: returns None and prints why."""
    try:
        return raw_content(title, lang)
    except Exception as e:
        print(f"  ! {title}: {e}")
        return None

#### `write_pages()`
Rewrites `pages.jsonl` through a temporary file, so a crash can't leave it half-written.

In [23]:
def write_pages(pages, path):
    """Rewrite pages.jsonl atomically (through a temp file), so a crash can't leave it half-written."""
    tmp = path.with_suffix(".jsonl.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for rec in pages.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    os.replace(tmp, path)

#### `update()`
The whole update.

In [24]:
def update(name, dry_run=False, skip_new=False, workers=4):
    """Compare saved revids with Wikipedia and re-download only what changed (see module doc)."""
    out_dir = corpus_dir(name)
    pages_path = out_dir / "pages.jsonl"
    config = json.loads((out_dir / "config.json").read_text(encoding="utf-8"))
    lang, with_links = config["lang"], config["with_links"]
    wiki = client(lang)
    pages = load_pages(pages_path)
    print(f"loaded {len(pages)} pages from {pages_path}")

    print("1) checking versions")
    meta = page_meta(list(pages), lang)
    backfilled = backfill(pages, meta)
    for title, rec in pages.items():  # older records were saved without coordinates
        if "coordinates" not in rec:
            rec["coordinates"] = meta[title]["coordinates"] if meta[title] else None

    changed, renamed, deleted = [], [], []
    for title, rec in pages.items():
        m = meta[title]
        if m is None:
            if rec.get("status") != "missing":
                deleted.append(title)
        elif m["title"] != title:
            renamed.append((title, m))
        elif m["revid"] != rec.get("revid"):
            changed.append((title, m))

    # Older records have no raw `content` (needed by export_corpus.py). For unchanged pages the
    # current text is the saved revision, so fetching it now is consistent.
    stale = {t for t, _ in changed} | {t for t, _ in renamed} | set(deleted)
    no_content = [t for t, rec in pages.items() if "content" not in rec and t not in stale and meta[t]]
    added = 0
    if no_content and not dry_run:
        print(f"  adding raw content to {len(no_content)} older records (saved every {CHECKPOINT}, so a stop loses little)")
        with ThreadPoolExecutor(min(workers, 2)) as pool:  # one request per page: go easy on the rate limit
            for i in range(0, len(no_content), CHECKPOINT):
                chunk = no_content[i:i + CHECKPOINT]
                for t, content in zip(chunk, pool.map(lambda t: try_raw_content(t, lang), chunk)):
                    if content is not None:  # failed pages stay without content; the next run retries them
                        pages[t]["content"] = content
                        added += 1
                write_pages(pages, pages_path)
                print(f"    {i + len(chunk)}/{len(no_content)}")

    new = []
    if not skip_new:
        print("2) re-scanning categories for new pages")
        titles = titles_from_config(wiki, config)
        known = set(pages) | {m["title"] for m in meta.values() if m}
        candidates = [t for t in titles if t not in known]
        seen = set(known)
        for t, m in page_meta(candidates, lang).items():
            if m and m["title"] not in seen:
                seen.add(m["title"])
                new.append((m, titles[t]))

    print(f"\nchanged: {len(changed)}   renamed: {len(renamed)}   deleted: {len(deleted)}   "
          f"new: {len(new) if not skip_new else 'skipped'}   backfilled: {backfilled}   "
          f"content added: {added}/{len(no_content)}")
    for t, m in changed[:20]:
        print(f"  ~ {t}  (edited {m['last_edited']})")
    for t, m in renamed:
        print(f"  > {t}  ->  {m['title']}")
    for t in deleted:
        print(f"  - {t}")
    for m, _ in new[:20]:
        print(f"  + {m['title']}")
    if dry_run:
        print("\ndry run: nothing written")
        return

    # Re-download changed + renamed pages, add new ones
    now = datetime.now(timezone.utc).isoformat(timespec="seconds")
    log = []
    to_fetch = [(m, pages[t]["source"]) for t, m in changed]
    to_fetch += [(m, pages[t]["source"]) for t, m in renamed if m["title"] not in pages]
    to_fetch += new
    old_revid = {t: pages[t].get("revid") for t, _ in changed}
    fetched = {}
    if to_fetch:
        print(f"\n3) fetching {len(to_fetch)} pages")
        fetch_many(wiki, to_fetch, with_links, workers, lambda r: fetched.__setitem__(r["title"], r))

    for t, m in changed:
        if m["title"] in fetched:
            pages[t] = fetched.pop(m["title"])
            log.append({"change": "updated", "title": t, "old_revid": old_revid[t], "new_revid": m["revid"]})
    for t, m in renamed:
        del pages[t]
        if m["title"] in fetched:
            pages[m["title"]] = fetched.pop(m["title"])
        log.append({"change": "renamed", "title": t, "new_title": m["title"]})
    for t in deleted:
        pages[t]["status"] = "missing"
        log.append({"change": "deleted", "title": t})
    for title, rec in fetched.items():  # what's left is new pages
        pages[title] = rec
        log.append({"change": "new", "title": title, "new_revid": rec["revid"]})

    write_pages(pages, pages_path)

    with (out_dir / "changes.jsonl").open("a", encoding="utf-8") as f:
        for entry in log:
            f.write(json.dumps({"time": now, **entry}, ensure_ascii=False) + "\n")
    print(f"\nsaved {len(pages)} pages; {len(log)} changes logged to changes.jsonl")

---
# Part 4: Filter
The category trees also bring pages that are *related* to the war but not military (sportspeople, songs,
reactions, sanctions...). Every category pages came from was reviewed: **84 kept, 53 removed**.

A page from a removed category is still kept (**rescued**) if it also belongs to a kept category,
or if it is a person with a military role (*… generals*, *… military personnel*, *GRU officers*, ...).
The original corpus is not touched; the filter writes a new one, with a report of what was removed and rescued.

#### The decisions
The categories whose pages are removed; every other category is kept.

In [25]:
REMOVE = [
    '2018 Moscow–Constantinople schism',
    'Anti-Maidan',
    'Battles involving the Wagner Group',
    'China and the Russian invasion of Ukraine',
    'Civilians killed in the Russo-Ukrainian war',
    'Disinformation in the Russian invasion of Ukraine',
    'Events affected by the 2014 Russian annexation of Crimea',
    'Events affected by the Russian invasion of Ukraine',
    'Events affected by the War in Donbas',
    'Genocide and the Russian invasion of Ukraine',
    'Grain and the Russian invasion of Ukraine',
    'Impacts of the Russian invasion of Ukraine',
    'Impacts of the Russo-Ukrainian war',
    'Incitement to genocide of Ukrainians',
    'Internet memes related to the Russian invasion of Ukraine',
    'Massacres committed by the Wagner Group',
    'Media and the Russo-Ukrainian war',
    'Military symbols used during the Russo-Ukrainian war',
    'Non-fiction works about the Russian invasion of Ukraine',
    'Non-fiction works about the Russo-Ukrainian war',
    'Nuclear technology and the Russian invasion of Ukraine',
    'People of the 2014 Russian annexation of Crimea',
    'People of the 2014 pro-Russian unrest in Ukraine',
    'People of the Russian invasion of Ukraine',
    'Pro-Ukrainian people of the Russian invasion of Ukraine',
    'Propaganda in Russia related to the Russian invasion of Ukraine',
    'Propaganda in Ukraine related to the Russian invasion of Ukraine',
    'Propaganda in the Russo-Ukrainian war',
    'Reactions to the Russian invasion of Ukraine',
    'Reactions to the Russo-Ukrainian war',
    'Russian activists against the Russian invasion of Ukraine',
    'Russian elections in Ukraine',
    'Russian invasion of Ukraine films',
    'Russian invasion of Ukraine in popular culture',
    'Russian invasion of Ukraine legal issues',
    'Russo-Ukrainian war films',
    'Russo-Ukrainian war in fiction',
    'Russo-Ukrainian war memorials',
    'Sanctions and boycotts during the Russo-Ukrainian war',
    'Songs about the Russian invasion of Ukraine',
    'Spillover of the Russian invasion of Ukraine',
    'Torture in the Russo-Ukrainian war',
    'Ukrainian collaborators with Russia',
    'Ukrainian sportspeople in Russia during the Russo-Ukrainian war',
    'United Kingdom and the Russian invasion of Ukraine',
    'United States and the Russian invasion of Ukraine',
    'Wagner Group',
    'Wagner Group activities by country',
    'War correspondents of the Russo-Ukrainian war',
    'Works about the 2014 Russian annexation of Crimea',
    'Works about the Russian invasion of Ukraine',
    'Works about the Russo-Ukrainian war',
    'Works about the war in Donbas',
]
KEEP = [
    '2014 Russian annexation of Crimea',
    '2014 pro-Russian unrest in Ukraine',
    'Aerial warfare in the Russian invasion of Ukraine',
    'Army corps of Ukraine in the Russo-Ukrainian war',
    'Attacks in Russia during the Russo-Ukrainian war (2022–present)',
    'Battle of Bakhmut',
    'Battle of Kharkiv (2022)',
    'Battles of the Eastern front of the Russian invasion of Ukraine',
    'Battles of the Northern front of the Russian invasion of Ukraine',
    'Battles of the Southern front of the Russian invasion of Ukraine',
    'Battles of the war in Donbas',
    'Belarus in the Russo-Ukrainian war',
    'Belarusian and Russian partisan movement (2022–present)',
    'Belarusian involvement in the Russian invasion of Ukraine',
    'Bombardments of the Russian invasion of Ukraine',
    'Buildings and structures destroyed during the Russian invasion of Ukraine',
    'Campaigns of the Russian invasion of Ukraine',
    'Civilians killed in the Russian invasion of Ukraine',
    'Crimea in the Russian invasion of Ukraine',
    'Crimea in the Russo-Ukrainian war',
    'Cyberattacks during the Russo-Ukrainian war',
    'Foreign involvement in the Russian invasion of Ukraine',
    'Foreign involvement in the Russo-Ukrainian war',
    'Foreign volunteers in the Russian invasion of Ukraine',
    'Malaysia Airlines Flight 17',
    'Military equipment of the Russian invasion of Ukraine',
    'Military equipment of the Russo-Ukrainian war',
    'Military operations of the Russian invasion of Ukraine',
    'Military operations of the Russian invasion of Ukraine in 2024',
    'Military operations of the Russian invasion of Ukraine in 2025',
    'Military operations of the Russo-Ukrainian war',
    'Military operations of the war in Donbas',
    'Military units and formations of Russia in the Russian invasion of Ukraine',
    'Military units and formations of Ukraine in the Russian invasion of Ukraine',
    'Military units and formations of the Russian invasion of Ukraine',
    'Naval warfare in the Russian invasion of Ukraine',
    'North Korean involvement in the Russian invasion of Ukraine',
    'Northern front of the Russian invasion of Ukraine',
    'Nuclear power plants and the Russian invasion of Ukraine',
    'Oblasts of Ukraine',
    'Organizations of the Russo-Ukrainian war',
    'People of the Wagner Group',
    'People of the war in Donbas',
    'Populated places destroyed during the Russian invasion of Ukraine',
    'Prelude to the Russian invasion of Ukraine',
    'Religious buildings and structures destroyed during the Russian invasion of Ukraine',
    'Russian airstrikes during the Russian invasion of Ukraine',
    'Russian annexations during the Russo-Ukrainian war',
    'Russian invasion of Ukraine by city',
    'Russian invasion of Ukraine orders of battle',
    'Russian occupation of Ukraine',
    'Russian referendums in Ukraine',
    'Russian war crimes in Ukraine',
    'Russo-Ukrainian cyberwarfare',
    'Russo-Ukrainian war',
    'Russo-Ukrainian war (2022–present)',
    'Russo-Ukrainian war peace process',
    'Russo-Ukrainian war related lists',
    'Russo-Ukrainian war truces',
    'Ships involved in the Russian invasion of Ukraine',
    'Ships involved in the Russo-Ukrainian war',
    'Shipwrecks of the Russian invasion of Ukraine',
    'Siege of Mariupol',
    'Southern front of the Russian invasion of Ukraine',
    'Theatres of operation of the Russian invasion of Ukraine',
    'Timelines of the Russian invasion of Ukraine',
    'Timelines of the Russo-Ukrainian war',
    'Timelines of the war in Donbas',
    'Torture in the Russian invasion of Ukraine',
    'Ukrainian airstrikes during the Russian invasion of Ukraine',
    'Ukrainian collaboration with Russia',
    'Ukrainian military personnel killed during the Battle of Bakhmut',
    'Ukrainian military personnel killed in the Kyiv offensive',
    'Ukrainian resistance during the Russian invasion of Ukraine',
    'Use of cluster munition during the Russian invasion of Ukraine',
    'Wagner Group activities in Ukraine',
    'War crimes during the Russian invasion of Ukraine',
    'War crimes in the Russo-Ukrainian war',
    'War in Donbas',
    'Women in the Russian invasion of Ukraine',
    'Women in the Russo-Ukrainian war',
    'seed',
    'seed-file:recommended_forces_battles_places.txt',
    'seed-file:team_DATA_json_titles.txt',
]

# filter_corpus() reads the decisions from this table
os.makedirs('filters', exist_ok=True)
with open('filters/category_review.csv', 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.writer(f)
    w.writerow(['category', 'decision'])
    w.writerows([(c, 'keep') for c in KEEP] + [(c, 'remove') for c in REMOVE])

#### Constants
Categories that mark a person with a military role, and look-alikes to ignore.

In [26]:
MILITARY_ROLE = re.compile(r"military personnel|\b(generals|admirals|colonels|officers)\b|naval commanders", re.I)
NOT_MILITARY = re.compile(r"Military World Games|Prosecutors general|police", re.I)

#### `strip()`
`Category:X` → `X`.

In [27]:
def strip(category):
    return category.replace("Category:", "", 1)

#### `load_decisions()`
Reads the decision table.

In [28]:
def load_decisions(path):
    """category_review.csv -> {category: "keep" | "remove"}."""
    with open(path, encoding="utf-8-sig") as f:
        return {row["category"]: row["decision"] for row in csv.DictReader(f)}

#### `classify()`
Keep, rescued (and why), or remove, for one page.

In [29]:
def classify(page, decisions):
    """-> ("keep" | "rescued" | "remove", reason)."""
    src = strip(page["source"])
    decision = decisions.get(src)
    if decision is None:
        return "keep", f"not in review table: {src}"
    if decision == "keep":
        return "keep", src
    rescue = [strip(c) for c in page["categories"] if decisions.get(strip(c)) == "keep"]
    if rescue:
        return "rescued", rescue[0]
    role = [strip(c) for c in page["categories"] if MILITARY_ROLE.search(c) and not NOT_MILITARY.search(c)]
    if role:
        return "rescued", f"military role: {role[0]}"
    return "remove", src

#### `filter_corpus()`
Writes the filtered corpus and a report (`summaries/<out>/`).

In [30]:
def filter_corpus(name, out, review="filters/category_review.csv"):
    """Write the pages that pass the review decisions to data/<out>/ and a report to summaries/<out>/."""
    decisions = load_decisions(review)
    pages = [p for p in load_pages(corpus_dir(name) / "pages.jsonl").values() if p.get("status") != "missing"]
    results = [(p, *classify(p, decisions)) for p in pages]

    out_dir = corpus_dir(out)
    out_dir.mkdir(parents=True, exist_ok=True)
    with (out_dir / "pages.jsonl").open("w", encoding="utf-8") as f:
        for p, status, _ in results:
            if status != "remove":
                f.write(json.dumps(p, ensure_ascii=False) + "\n")

    report_dir = Path("summaries") / out
    report_dir.mkdir(parents=True, exist_ok=True)
    removed = [(p, why) for p, status, why in results if status == "remove"]
    rescued = [(p, why) for p, status, why in results if status == "rescued"]
    unknown = Counter(why for _, status, why in results if why.startswith("not in review"))
    with (report_dir / "removed.csv").open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(["title", "removed_category", "url"])
        for p, why in sorted(removed, key=lambda r: (r[1], r[0]["title"])):
            w.writerow([p["title"], why, p["url"]])

    kept = len(pages) - len(removed)
    md = [
        f"# Filter report: `{name}` → `{out}`",
        f"Decisions: `{review}`.",
        "",
        "| | Pages |", "|---|---|",
        f"| Before | {len(pages):,} |",
        f"| Removed | {len(removed):,} |",
        f"| Rescued (source removed, but also in a kept category) | {len(rescued):,} |",
        f"| **After** | **{kept:,}** |",
        "",
        "## Removed, by category",
        "| Category | Pages |", "|---|---|",
        *[f"| {c} | {n} |" for c, n in Counter(why for _, why in removed).most_common()],
        "",
        "## Rescued pages (kept because of another category)",
        "| Page | Came from (removed) | Kept by |", "|---|---|---|",
        *[f"| {p['title']} | {strip(p['source'])} | {why} |" for p, why in sorted(rescued, key=lambda r: r[0]["title"])],
    ]
    if unknown:
        md += ["", "## Sources not in the review table (kept)", "| Source | Pages |", "|---|---|",
               *[f"| {w.split(': ', 1)[1]} | {n} |" for w, n in unknown.most_common()]]
    (report_dir / "filter_report.md").write_text("\n".join(md) + "\n", encoding="utf-8")

    print(f"{len(pages)} pages -> {kept} kept ({len(rescued)} rescued), {len(removed)} removed")
    if unknown:
        print(f"warning: {sum(unknown.values())} pages from {len(unknown)} sources not in the review table were kept")
    print(f"wrote {out_dir / 'pages.jsonl'} and {report_dir}/")

---
# Part 5: Export
`DATA.json` for the next step: `{title: text}` with each page's raw text (`content`), written with
`json.dump` defaults exactly like `examples/DATA.json`.

#### `export_data_json()`
`pages.jsonl` → `data/<name>/DATA.json`.

In [31]:
def export_data_json(name, out_file="DATA.json"):
    """pages.jsonl -> data/<name>/DATA.json in the exact format of examples/DATA.json."""
    pages = [p for p in load_pages(corpus_dir(name) / "pages.jsonl").values() if p.get("status") != "missing"]
    no_content = [p["title"] for p in pages if "content" not in p]
    if no_content:
        raise ValueError(f"{len(no_content)} pages have no `content` (e.g. {no_content[:3]}); "
                         f"run update_corpus.py on the source corpus first")
    data = {p["title"]: p["content"] for p in pages}
    path = corpus_dir(name) / out_file
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f)
    print(f"wrote {path}: {len(data)} pages, {path.stat().st_size / 1e6:.1f} MB")
    return path

---
# Part 6: Run

### A. Quick demo (about 2 minutes)
A small corpus: 2014–2021 battles plus songs about the war (so the filter has something to remove).

In [32]:
build('demo', depth=0, with_links=False, seeds=['Battle of Bakhmut'],
      categories=['Category:Battles of the war in Donbas', 'Category:Songs about the Russian invasion of Ukraine'])
filter_corpus('demo', 'demo_military')
path = export_data_json('demo_military')

1) collecting titles


  Category:Battles of the war in Donbas (depth 0): 1 categories, 27 pages so far


  Category:Songs about the Russian invasion of Ukraine (depth 0): 1 categories, 47 pages so far
2) checking versions of 47 pages


3) fetching 47 pages (0 already saved)


  25/47  (100s)


done: 47 new pages -> data\demo\pages.jsonl (total 47)
47 pages -> 29 kept (2 rescued), 18 removed
wrote data\demo_military\pages.jsonl and summaries\demo_military/
wrote data\demo_military\DATA.json: 29 pages, 0.3 MB


The result has exactly the format of `examples/DATA.json`, and the NER notebook's paragraph split works on it:

In [33]:
data = json.load(open(path, encoding='utf-8'))
example = json.loads(requests.get('https://raw.githubusercontent.com/arielax-212/Corpus2GeoRAG/main/examples/DATA.json').text)
print(type(data).__name__, len(data), 'pages')
print('same text as examples/DATA.json:', raw_content('Berdiansk port attack') == example['2022 Berdiansk port attack'])

paragraphs = [t for t in data['Battle of Bakhmut'].split('\n') if not t.startswith('==') and len(t) >= 5]
print(len(paragraphs), 'NER paragraphs from Battle of Bakhmut')

dict 29 pages


same text as examples/DATA.json: True
97 NER paragraphs from Battle of Bakhmut


### B. The full corpus
Set `RUN_FULL = True` to build the real corpus. The first build downloads every page (1–2 hours, depending on
Wikipedia's rate limit). After that, `update()` only checks versions (a few minutes) and re-downloads the pages
that changed, so a weekly refresh is short. Filter and export take seconds.

In [34]:
RUN_FULL = False

if RUN_FULL:
    for f in ['team_DATA_json_titles.txt', 'recommended_forces_battles_places.txt']:   # seed lists from the repo
        os.makedirs('seeds', exist_ok=True)
        open(f'seeds/{f}', 'w', encoding='utf-8').write(
            requests.get(f'https://raw.githubusercontent.com/arielax-212/Corpus2GeoRAG/main/CollectCorpus/seeds/{f}').text)

    build('ukraine_war', depth=2,
          categories=['Category:Russo-Ukrainian war (2022–present)',
                      'Category:Russo-Ukrainian war',
                      'Category:Military operations of the Russian invasion of Ukraine@3',
                      'Category:Military units and formations of the Russian invasion of Ukraine@3',
                      'Category:Military equipment of the Russian invasion of Ukraine@3',
                      'Category:Battles of the war in Donbas@0',
                      'Category:Oblasts of Ukraine@0'],
          seeds=['Ukraine', 'Russia', 'Russo-Ukrainian war',
                 'Annexation of Crimea by the Russian Federation', 'War in Donbas'],
          seed_files=['seeds/team_DATA_json_titles.txt', 'seeds/recommended_forces_battles_places.txt'])
    update('ukraine_war', skip_new=True)          # later runs: only this, then filter + export
    filter_corpus('ukraine_war', 'ukraine_war_military')
    export_data_json('ukraine_war_military')      # -> data/ukraine_war_military/DATA.json